## Install the required libraries (Skip if already installed)

Install Ultralytics
(Install in the notebook kernel)

In [ ]:
pip install ultralytics roboflow
pip install ultralytics

Install Optuna
(Install in the notebook kernel)

In [ ]:
pip install optuna

If you have a version of PyTorch without GPU support it must be uninstalled (If you do not have any version of PyTorch, ignore the following line)
(Run in the Kernel console)

In [ ]:
pip uninstall torch torchvision torchaudio

Install the PyTorch version that matches your PC CUDA version

In [ ]:
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

## Verify GPU availability and Ultralytics

In [1]:
!nvidia-smi

Mon Jun 29 10:37:08 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 591.91                 Driver Version: 591.91         CUDA Version: 13.1     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 5060 ...  WDDM  |   00000000:01:00.0 Off |                  N/A |
| N/A   44C    P8              7W /   42W |       0MiB /   8151MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import torch

def check_cuda():
    print("=== PyTorch CUDA Diagnostic ===\n")

    # 1. Is CUDA available?
    cuda_available = torch.cuda.is_available()
    print(f"CUDA available: {cuda_available}")

    # 2. PyTorch build info
    print(f"PyTorch version: {torch.__version__}")
    print(f"CUDA version (torch): {torch.version.cuda}")

    if not cuda_available:
        print("\n[FAIL] CUDA is not available.")
        return

    # 3. GPU details
    device_count = torch.cuda.device_count()
    print(f"\nNumber of GPUs: {device_count}")

    for i in range(device_count):
        print(f"\n--- GPU {i} ---")
        print(f"Name: {torch.cuda.get_device_name(i)}")
        print(f"Capability: {torch.cuda.get_device_capability(i)}")
        print(f"Memory Allocated: {torch.cuda.memory_allocated(i)} bytes")
        print(f"Memory Reserved: {torch.cuda.memory_reserved(i)} bytes")

    # 4. Actual computation test (this is the only *real* proof)
    try:
        device = torch.device("cuda:0")
        x = torch.rand(3, 3).to(device)
        y = torch.rand(3, 3).to(device)
        z = x @ y

        print("\n[OK] Tensor computation on GPU succeeded.")
        print(f"Result device: {z.device}")

    except Exception as e:
        print("\n[FAIL] CUDA exists but computation failed.")
        print(f"Error: {e}")


if __name__ == "__main__":
    check_cuda()

=== PyTorch CUDA Diagnostic ===

CUDA available: True
PyTorch version: 2.12.0.dev20260324+cu128
CUDA version (torch): 12.8

Number of GPUs: 1

--- GPU 0 ---
Name: NVIDIA GeForce RTX 5060 Laptop GPU
Capability: (12, 0)
Memory Allocated: 0 bytes
Memory Reserved: 0 bytes

[OK] Tensor computation on GPU succeeded.
Result device: cuda:0


In [3]:
import ultralytics
ultralytics.checks()

Ultralytics 8.4.26  Python-3.12.10 torch-2.12.0.dev20260324+cu128 CUDA:0 (NVIDIA GeForce RTX 5060 Laptop GPU, 8151MiB)
Setup complete  (24 CPUs, 31.6 GB RAM, 734.1/924.4 GB disk)


## Dataset Unzip

In [4]:
from pathlib import Path
import zipfile

# Paths
ROOT = Path.cwd().parent
DATA_DIR = ROOT / "0_Dataset"

DATASET_PATH_ZIP = DATA_DIR / "License_Plate_Recognition_yolov8.zip"
EXTRACT_DIR = DATA_DIR / "License_Plate_Recognition_yolov8"

In [5]:
# Unzip
with zipfile.ZipFile(DATASET_PATH_ZIP, "r") as zip_ref:
    zip_ref.extractall(EXTRACT_DIR)

print("Extracted to:", EXTRACT_DIR)

Extracted to: c:\Users\lucas\Desktop\car_plates_recognition\0_Dataset\License_Plate_Recognition_yolov8


## Dataset inspection

In [6]:
from pathlib import Path

# Path
ROOT = Path.cwd().parent

DATASET_PATH = ROOT / "0_Dataset" / "License_Plate_Recognition_yolov8"

In [7]:
import os
os.listdir(DATASET_PATH)

['data.yaml',
 'README.dataset.txt',
 'README.roboflow.txt',
 'test',
 'train',
 'valid']

In [11]:
DATASET_PATH_YAML = ROOT / "0_Dataset" / "License_Plate_Recognition_yolov8" / "data.yaml"

with open(DATASET_PATH_YAML, 'r') as f:
    print(f.read())

train: ../train/images
val: ../valid/images
test: ../test/images

nc: 1
names: ['License_Plate']

roboflow:
  workspace: roboflow-universe-projects
  project: license-plate-recognition-rxg4e
  version: 11
  license: CC BY 4.0
  url: https://universe.roboflow.com/roboflow-universe-projects/license-plate-recognition-rxg4e/dataset/11


In [12]:
import os
from collections import defaultdict

def analyze_yolo_roboflow_dataset(dataset_path):
    splits = ['train', 'valid', 'test']
    ext_img = ('.jpg', '.jpeg', '.png')

    total_imgs = 0
    split_counts = {}
    class_counts = defaultdict(int)
    all_classes = set()

    print(f"\n Analyzing dataset in: {dataset_path}\n")

    for split in splits:
        images_dir = os.path.join(dataset_path, split, "images")
        labels_dir = os.path.join(dataset_path, split, "labels")

        if not os.path.exists(images_dir):
            print(f" Folder not found: {images_dir}")
            continue

        images = [f for f in os.listdir(images_dir) if f.lower().endswith(ext_img)]
        split_counts[split] = len(images)
        total_imgs += len(images)

        for img in images:
            base_name = os.path.splitext(img)[0]
            label_file = base_name + '.txt'
            label_path = os.path.join(labels_dir, label_file)
            if os.path.exists(label_path):
                with open(label_path, 'r') as f:
                    lines = f.readlines()
                    for line in lines:
                        cls_id = line.strip().split()[0]
                        class_counts[cls_id] += 1
                        all_classes.add(cls_id)

    print(" Dataset Statistics:")
    print(f"   - Total images: {total_imgs}")
    for split in splits:
        count = split_counts.get(split, 0)
        pct = (count / total_imgs * 100) if total_imgs > 0 else 0
        print(f"   - {split.capitalize():<6}: {count} images ({pct:.2f}%)")

    print("\n Classes found:")
    if all_classes:
        sorted_classes = sorted(all_classes, key=lambda x: int(x))
        for cls_id in sorted_classes:
            print(f"   - Class {cls_id}: {class_counts[cls_id]} images")
    else:
        print("   - No classes found (check the .txt files)")

analyze_yolo_roboflow_dataset(DATASET_PATH)


 Analyzing dataset in: c:\Users\lucas\Desktop\car_plates_recognition\0_Dataset\License_Plate_Recognition_yolov8

 Dataset Statistics:
   - Total images: 10125
   - Train : 7057 images (69.70%)
   - Valid : 2048 images (20.23%)
   - Test  : 1020 images (10.07%)

 Classes found:
   - Class 0: 10637 images


## Hyperparameter Search

In [ ]:
import optuna 
from ultralytics import YOLO
import torch
import os
import joblib
import yaml

# Directory to save the results
SAVE_DIR = "optuna_yolo_runs"
os.makedirs(SAVE_DIR, exist_ok=True)

# Check if CUDA is available
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f" Using device: {device}")

def objective(trial):
    trial_params = {
        'lr0': trial.suggest_float('lr0', 1e-5, 1e-1, log=True),
        'lrf': trial.suggest_float('lrf', 0.01, 1.0),
        'momentum': trial.suggest_float('momentum', 0.6, 0.98),
        'weight_decay': trial.suggest_float('weight_decay', 0.0001, 0.01),
        'warmup_epochs': trial.suggest_float('warmup_epochs', 0.0, 5.0),
        'box': trial.suggest_float('box', 0.02, 0.2),
        'cls': trial.suggest_float('cls', 0.2, 0.5),
        'dfl': trial.suggest_float('dfl', 0.5, 1.0),
        'hsv_h': trial.suggest_float('hsv_h', 0.0, 0.1),
        'hsv_s': trial.suggest_float('hsv_s', 0.0, 0.9),
        'hsv_v': trial.suggest_float('hsv_v', 0.0, 0.9),
        'flipud': trial.suggest_float('flipud', 0.0, 0.5),
        'fliplr': trial.suggest_float('fliplr', 0.0, 0.5),
        'mosaic': trial.suggest_float('mosaic', 0.0, 1.0),
        'mixup': trial.suggest_float('mixup', 0.0, 1.0)
    }

    # Create a unique name for this run
    run_dir = os.path.join(SAVE_DIR, f"trial_{trial.number}")
    os.makedirs(run_dir, exist_ok=True)

    try:
        model = YOLO("yolov8s.pt")

        results = model.train(
            data="dataset/data.yaml", # Change to YOUR dataset path
            epochs=20,
            imgsz=224,
            device=device,  # force CUDA usage if available
            project=SAVE_DIR,
            name=f"trial_{trial.number}",
            exist_ok=True,
            **trial_params
        )

        metrics = model.val(device=device)
        map50 = metrics.box.map50

        return map50

    except Exception as e:
        print(f" Error in trial {trial.number}: {e}")
        return 0.0  # Return a low value if an error occurs

# Create study
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=10)  # you can increase the number if you have a powerful GPU

# Show best hyperparameters
print("\n Best trial:")
print(study.best_trial.params)

# Save the best hyperparameters as a .yaml file
with open("best_hyp.yaml", "w") as f:
    yaml.dump(study.best_params, f)


In [ ]:
import optuna.visualization as vis

vis.plot_optimization_history(study).show()
vis.plot_param_importances(study).show()

In [2]:
import yaml

# Dictionary with the best hyperparameters found
best_trial = {
    'lr0': 2.1317467504762106e-05,
    'lrf': 0.030072847945439234,
    'momentum': 0.8821688165695634,
    'weight_decay': 0.0006166326723553464,
    'warmup_epochs': 4.557648004059905,
    'box': 0.10023029534243469,
    'cls': 0.25460192577090784,
    'dfl': 0.6724280792216191,
    'hsv_h': 0.0509936944012377,
    'hsv_s': 0.4079405015001844,
    'hsv_v': 0.5157638419504308,
    'flipud': 0.04637695520824986,
    'fliplr': 0.0888450285798808,
    'mosaic': 0.4986086074532513,
    'mixup': 0.08306650293191398
}

# Save as YAML file
yaml_filename = "best_hyperparameters.yaml"

with open(yaml_filename, "w") as file:
    yaml.dump(best_trial, file, sort_keys=False)

print(f"File '{yaml_filename}' generated successfully.")


File 'best_hyperparameters.yaml' generated successfully.


## Train the Model (Using the best hyperparameters)

In [ ]:
from ultralytics import YOLO
import yaml

# Load the model
model_bh = YOLO("yolov8s.pt")

# Check if CUDA is available
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f" Using device: {device}")

# Read the YAML file with hyperparameters
with open("best_hyperparameters.yaml", "r") as f:
    hyp_params = yaml.safe_load(f)

# Train the model, passing the hyperparameters as arguments
results = model_bh.train(
    data="/content/dataset/data.yaml",
    epochs=270,
    imgsz=224,
    device=device,  # force CUDA usage if available
    **hyp_params  # Best Hypers
)

# Display model metrics
print("Model metrics:")
metrics = model_bh.val()

Visualize the metrics results

In [ ]:
metrics = model_bh.val()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Path to the CSV file
metrics_path = "/content/runs/detect/train/results.csv"
df = pd.read_csv(metrics_path)

# ---- LOSSES IN PAIRS ----
loss_pairs = [
    ("train/box_loss", "val/box_loss", "Box Loss"),
    ("train/cls_loss", "val/cls_loss", "Cls Loss"),
    ("train/dfl_loss", "val/dfl_loss", "DFL Loss")
]

for train_col, val_col, title in loss_pairs:
    if train_col in df.columns and val_col in df.columns:
        fig, axs = plt.subplots(1, 2, figsize=(12, 4), sharey=True)

        axs[0].plot(df["epoch"], df[train_col], color='blue')
        axs[0].set_title(f"Train {title}")
        axs[0].set_xlabel("Epoch")
        axs[0].set_ylabel("Loss")
        axs[0].grid(True)

        axs[1].plot(df["epoch"], df[val_col], color='orange')
        axs[1].set_title(f"Validation {title}")
        axs[1].set_xlabel("Epoch")
        axs[1].grid(True)

        plt.suptitle(f"{title} Comparison")
        plt.tight_layout()
        plt.show()

# ---- PERFORMANCE METRICS IN PAIRS ----
metric_pairs = [
    ("metrics/precision(B)", "metrics/recall(B)", "Precision vs Recall"),
    ("metrics/mAP50(B)", "metrics/mAP50-95(B)", "MAP50 vs MAP50-95")
]

for col1, col2, title in metric_pairs:
    if col1 in df.columns and col2 in df.columns:
        fig, axs = plt.subplots(1, 2, figsize=(12, 4), sharey=True)

        axs[0].plot(df["epoch"], df[col1], color='green')
        axs[0].set_title(col1.split("/")[-1])
        axs[0].set_xlabel("Epoch")
        axs[0].set_ylabel("Value")
        axs[0].grid(True)

        axs[1].plot(df["epoch"], df[col2], color='purple')
        axs[1].set_title(col2.split("/")[-1])
        axs[1].set_xlabel("Epoch")
        axs[1].grid(True)

        plt.suptitle(title)
        plt.tight_layout()
        plt.show()